In [1]:
import numpy as np
from torchvision import datasets, transforms
import torch
from time import time
from torch import nn, optim

In [2]:
DOWNLOAD = False
# train_path = r"/scratch/mnist/train"
# test_path = r"/scratch/mnist/test"
train_path = r"C:\Users\hbse\projects\data\metrics\train"
test_path = r"C:\Users\hbse\projects\data\metrics\test"

In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,),)])
trainset = datasets.MNIST(train_path, download=DOWNLOAD, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=len(trainset), shuffle=True)
# valset = datasets.MNIST(test_path, download=DOWNLOAD, train=False, transform=transform)
# valloader = torch.utils.data.DataLoader(valset, batch_size=len(trainset), shuffle=True)

In [7]:
print("Loading training dataset...", end="")
train_iter = iter(trainloader)
train_images, train_labels = next(train_iter)
print("Loaded.")
# print("Loading validation dataset...", end="")
# val_iter = iter(valloader)
# val_images, val_labels = next(val_iter)
# print("Loaded.")

train_images.shape, train_labels.shape#, val_images.shape, val_labels.shape

Loading training dataset...Loaded.


(torch.Size([60000, 1, 28, 28]), torch.Size([60000]))

In [8]:
IMG_PIXEL_X = 28
IMG_PIXEL_Y = 28
INPUT_SIZE = IMG_PIXEL_X*IMG_PIXEL_Y
HIDDEN_SIZES = [128, 64]
OUTPUT_SIZE = 10
N_MODELS = 3

# torch.manual_seed(0)

model = nn.Sequential(nn.Linear(INPUT_SIZE, HIDDEN_SIZES[0]),
                        nn.ReLU(),
                        nn.Linear(HIDDEN_SIZES[0], HIDDEN_SIZES[1]),
                        nn.ReLU(),
                        nn.Linear(HIDDEN_SIZES[1], OUTPUT_SIZE),
                        nn.LogSoftmax(dim=1))
# models

In [9]:
model_params = list(model.parameters())

In [10]:
[model_param.shape for model_param in model_params]

[torch.Size([128, 784]),
 torch.Size([128]),
 torch.Size([64, 128]),
 torch.Size([64]),
 torch.Size([10, 64]),
 torch.Size([10])]

#### get connections from layer 2 to 3

In [11]:
w23=model_params[2]

In [12]:
w23.grad

#### understand the effect of zero_grad()

In [14]:
optimizer = optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
loss_fn = nn.NLLLoss()
time0 = time()
# epochs = 15
epochs = 2
model.cuda()  # if cuda enabled, make sure images and labels are in cuda too
train_images, train_labels = train_images.cuda(), train_labels.cuda()  # if cuda enabled, make sure model is in cuda too
print(f"Model, device: {list(model.parameters())[0].device}. Images, device: {train_images.device}")
print(f"Start training...  Time (in minutes) ={(time()-time0)/60:.2f}")
for e in range(epochs):
    print(f"epoch {e}")
    print("grad before optimizer.zero_grad()\n", w23.grad)
    optimizer.zero_grad()  # Training pass  # todo: not sure why we need this
    print("grad after optimizer.zero_grad()\n", w23.grad)
    
    class_probs = model(train_images.reshape(-1, INPUT_SIZE))  # log probabilities
    loss = loss_fn(class_probs, train_labels)  # calculate the NLL loss

    print("grad after evaluating model and calculating loss\n", w23.grad)
    
    loss.backward()  # backpropagate model loss
    print("grad after loss.backward()\n", w23.grad)
    optimizer.step()  # optimizes model weights
    print(f"model: Epoch {e} - Training loss: {loss.item():.6f}", end="")
    print(f"  Time (in minutes)={(time()-time0)/60:.2f}")
print(f"\nTotal Training Time (in minutes)={(time()-time0)/60:.2f}\n")

Model, device: cuda:0. Images, device: cuda:0
Start training...  Time (in minutes) =0.00
epoch 0
grad before optimizer.zero_grad()
 tensor([[-4.2148e-05,  2.2787e-05,  1.8972e-06,  ..., -8.6714e-05,
          4.0349e-06, -3.6065e-05],
        [ 1.1761e-03,  8.1295e-04, -7.3342e-04,  ...,  1.0775e-04,
         -2.5145e-05,  1.2101e-03],
        [-7.1118e-05, -7.6140e-05, -3.9229e-07,  ..., -1.6814e-05,
         -4.8780e-07, -1.6014e-06],
        ...,
        [ 4.5300e-03,  4.7031e-03,  7.5081e-04,  ...,  3.3105e-04,
          1.8076e-04,  1.2347e-03],
        [-1.3896e-03, -5.7333e-04,  4.6587e-04,  ...,  2.0231e-03,
          7.5812e-05,  1.2274e-04],
        [-4.1143e-03,  4.5913e-03,  8.0838e-04,  ...,  1.1479e-03,
         -3.3767e-06,  3.2003e-03]], device='cuda:0')
grad after optimizer.zero_grad()
 None
grad after evaluating model and calculating loss
 None
grad after loss.backward()
 tensor([[-4.4409e-05,  1.9468e-05,  2.3168e-06,  ..., -8.8764e-05,
          4.0002e-06, -3.4462e

zero_grad() resets to `None` the `grad` attribute of the Tensor variable

In [15]:
class_probs.shape

torch.Size([60000, 10])

In [16]:
loss

tensor(2.3184, device='cuda:0', grad_fn=<NllLossBackward0>)

#### Re-running loss.backward() leads to an error

After computing the gradient with `loss.backward()`, we need to do `optimizer.zero_grad()` to run it again without error

In [17]:
w23.grad

tensor([[-4.8209e-05,  1.8682e-05,  2.4831e-06,  ..., -8.9884e-05,
          3.9966e-06, -3.4893e-05],
        [ 1.0820e-03,  7.4278e-04, -7.2464e-04,  ...,  9.7723e-05,
         -2.3651e-05,  1.1439e-03],
        [-7.4767e-05, -7.9043e-05, -2.2653e-07,  ..., -1.6354e-05,
         -4.8749e-07, -1.5042e-06],
        ...,
        [ 4.4320e-03,  4.5642e-03,  7.1203e-04,  ...,  2.1076e-04,
          1.7958e-04,  1.1268e-03],
        [-1.3638e-03, -5.4226e-04,  4.5532e-04,  ...,  1.9578e-03,
          7.5632e-05,  1.1671e-04],
        [-4.0824e-03,  4.5960e-03,  7.7855e-04,  ...,  1.0806e-03,
         -3.3194e-06,  3.0549e-03]], device='cuda:0')

In [18]:
loss.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [19]:
initial_w23 = w23.clone()

In [20]:
torch.sum(w23-initial_w23)

tensor(0., device='cuda:0', grad_fn=<SumBackward0>)

In [21]:
optimizer.zero_grad()  # resets tensor gradients
print("grad after optimizer.zero_grad()\n", w23.grad)

class_probs = model(train_images.reshape(-1, INPUT_SIZE))  # log probabilities
loss = loss_fn(class_probs, train_labels)  # calculate the NLL loss

loss.backward()  # backpropagate model loss
print("grad after loss.backward()\n", w23.grad)
print("weight changes", torch.sum(torch.abs(w23-initial_w23)))
print("try to run loss.backward()...")
loss.backward()  # backpropagate model loss
print("grad after loss.backward() again\n", w23.grad)  # doesn't print because re-running loss.backward yields error
optimizer.step()  # change model weights
print("grad after optimizer.step()\n", w23.grad)

grad after optimizer.zero_grad()
 None
grad after loss.backward()
 tensor([[-6.0446e-05,  6.3043e-06,  2.0523e-06,  ..., -9.3580e-05,
          3.9899e-06, -3.6059e-05],
        [ 1.0289e-03,  7.0876e-04, -7.1860e-04,  ...,  9.5664e-05,
         -2.3038e-05,  1.1046e-03],
        [-7.3283e-05, -7.6659e-05, -2.1824e-07,  ..., -1.5753e-05,
         -4.8722e-07, -1.1932e-06],
        ...,
        [ 4.3709e-03,  4.4787e-03,  6.8805e-04,  ...,  1.3817e-04,
          1.7885e-04,  1.0586e-03],
        [-1.3473e-03, -5.2269e-04,  4.4824e-04,  ...,  1.9174e-03,
          7.5521e-05,  1.1220e-04],
        [-4.0610e-03,  4.6033e-03,  7.5909e-04,  ...,  1.0399e-03,
         -3.1988e-06,  2.9593e-03]], device='cuda:0')
weight changes tensor(0., device='cuda:0', grad_fn=<SumBackward0>)
try to run loss.backward()...


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

If `loss.backward()` is run after `optimizer.step()` the same error happens:

In [22]:
optimizer.zero_grad()  # resets tensor gradients
print("grad after optimizer.zero_grad()\n", w23.grad)

class_probs = model(train_images.reshape(-1, INPUT_SIZE))  # log probabilities
loss = loss_fn(class_probs, train_labels)  # calculate the NLL loss
loss.backward()  # backpropagate model loss
print("grad after loss.backward()\n", w23.grad)
optimizer.step()  # change model weights
print("grad after optimizer.step()\n", w23.grad)
print("try to run loss.backward()...")
loss.backward()  # backpropagate model loss
print("grad after loss.backward() again\n", w23.grad)

grad after optimizer.zero_grad()
 None
grad after loss.backward()
 tensor([[-6.0446e-05,  6.3043e-06,  2.0523e-06,  ..., -9.3580e-05,
          3.9899e-06, -3.6059e-05],
        [ 1.0289e-03,  7.0876e-04, -7.1860e-04,  ...,  9.5664e-05,
         -2.3038e-05,  1.1046e-03],
        [-7.3283e-05, -7.6659e-05, -2.1824e-07,  ..., -1.5753e-05,
         -4.8722e-07, -1.1932e-06],
        ...,
        [ 4.3709e-03,  4.4787e-03,  6.8805e-04,  ...,  1.3817e-04,
          1.7885e-04,  1.0586e-03],
        [-1.3473e-03, -5.2269e-04,  4.4824e-04,  ...,  1.9174e-03,
          7.5521e-05,  1.1220e-04],
        [-4.0610e-03,  4.6033e-03,  7.5909e-04,  ...,  1.0399e-03,
         -3.1988e-06,  2.9593e-03]], device='cuda:0')
grad after optimizer.step()
 tensor([[-6.0446e-05,  6.3043e-06,  2.0523e-06,  ..., -9.3580e-05,
          3.9899e-06, -3.6059e-05],
        [ 1.0289e-03,  7.0876e-04, -7.1860e-04,  ...,  9.5664e-05,
         -2.3038e-05,  1.1046e-03],
        [-7.3283e-05, -7.6659e-05, -2.1824e-07,  

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

No problem computing loss before `optimizer.zero_grad()` and only after do `loss.backward()`:

In [23]:
class_probs = model(train_images.reshape(-1, INPUT_SIZE))  # log probabilities
loss = loss_fn(class_probs, train_labels)  # calculate the NLL loss

optimizer.zero_grad()  # resets tensor gradients
print("grad after optimizer.zero_grad()\n", w23.grad)

loss.backward()  # backpropagate model loss
print("grad after loss.backward()\n", w23.grad)
optimizer.step()  # change model weights
print("grad after optimizer.step()\n", w23.grad)

grad after optimizer.zero_grad()
 None
grad after loss.backward()
 tensor([[-6.4780e-05,  7.2850e-06,  1.8795e-06,  ..., -9.5679e-05,
          4.1252e-06, -3.5326e-05],
        [ 9.9000e-04,  7.1287e-04, -7.0624e-04,  ...,  1.0090e-04,
         -2.0570e-05,  1.0534e-03],
        [-7.3083e-05, -7.6219e-05, -2.0637e-07,  ..., -1.4780e-05,
         -4.8688e-07, -1.0311e-06],
        ...,
        [ 4.2845e-03,  4.3569e-03,  6.5465e-04,  ...,  3.8645e-05,
          1.7774e-04,  9.6484e-04],
        [-1.3239e-03, -4.9532e-04,  4.3811e-04,  ...,  1.8614e-03,
          7.5365e-05,  1.0509e-04],
        [-4.0302e-03,  4.6119e-03,  7.3168e-04,  ...,  9.8703e-04,
         -2.7864e-06,  2.8226e-03]], device='cuda:0')
grad after optimizer.step()
 tensor([[-6.4780e-05,  7.2850e-06,  1.8795e-06,  ..., -9.5679e-05,
          4.1252e-06, -3.5326e-05],
        [ 9.9000e-04,  7.1287e-04, -7.0624e-04,  ...,  1.0090e-04,
         -2.0570e-05,  1.0534e-03],
        [-7.3083e-05, -7.6219e-05, -2.0637e-07,  

#### Weight changes occur after `optimizer.step()`. `loss.backward()` only computes gradients but makes no changes

In [24]:
initial_w23 = w23.clone()
optimizer.zero_grad()  # resets tensor gradients
print("grad after optimizer.zero_grad()\n", w23.grad)

class_probs = model(train_images.reshape(-1, INPUT_SIZE))  # log probabilities
loss = loss_fn(class_probs, train_labels)  # calculate the NLL loss

loss.backward()  # backpropagate model loss
print("grad after loss.backward()\n", w23.grad)
print("sum abs weight changes", torch.sum(torch.abs(w23-initial_w23)))
optimizer.step()  # change model weights
print("\ngrad after optimizer.step()\n", w23.grad)
print("sum abs weight changes", torch.sum(torch.abs(w23-initial_w23)))

grad after optimizer.zero_grad()
 None
grad after loss.backward()
 tensor([[-7.5781e-05, -6.5035e-06,  1.1956e-06,  ..., -1.0007e-04,
          4.1413e-06, -3.6035e-05],
        [ 9.1682e-04,  6.8449e-04, -6.9851e-04,  ...,  1.0225e-04,
         -1.8730e-05,  9.8471e-04],
        [-7.2840e-05, -7.5350e-05, -1.0219e-07,  ..., -1.4187e-05,
         -4.8641e-07, -8.7376e-07],
        ...,
        [ 4.1825e-03,  4.2178e-03,  6.1415e-04,  ..., -7.3837e-05,
          1.7650e-04,  8.5224e-04],
        [-1.2947e-03, -4.6148e-04,  4.2527e-04,  ...,  1.7938e-03,
          7.5176e-05,  9.5875e-05],
        [-3.9955e-03,  4.6186e-03,  6.9720e-04,  ...,  9.2589e-04,
         -2.6868e-06,  2.6494e-03]], device='cuda:0')
sum abs weight changes tensor(0., device='cuda:0', grad_fn=<SumBackward0>)

grad after optimizer.step()
 tensor([[-7.5781e-05, -6.5035e-06,  1.1956e-06,  ..., -1.0007e-04,
          4.1413e-06, -3.6035e-05],
        [ 9.1682e-04,  6.8449e-04, -6.9851e-04,  ...,  1.0225e-04,
         